# Wildfire ConvLSTM — Kaggle GPU Training

**Before running:**
1. Set accelerator: `Settings → Accelerator → GPU T4 x2` (or GPU P100)
2. Add all `wildfire-drone-p1` … `wildfire-drone-pN` datasets as inputs
3. Run all cells top to bottom

In [ ]:
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
print('GPU count:', torch.cuda.device_count())
print('CUDA:', torch.version.cuda)

In [ ]:
# Clone repo and install deps
!git clone https://github.com/malihashar/wildfire-drone.git
%cd wildfire-drone
!pip install tqdm -q

In [ ]:
# Symlink all simulation files from every dataset part into one flat directory
import os, glob

SIM_DIR = '/kaggle/working/simulations'
os.makedirs(SIM_DIR, exist_ok=True)

all_pt = sorted(glob.glob('/kaggle/input/wildfire-drone-p*/*.pt'))
print(f'Found {len(all_pt)} .pt files across all dataset parts')

linked = 0
for src in all_pt:
    dst = os.path.join(SIM_DIR, os.path.basename(src))
    if not os.path.exists(dst):
        os.symlink(src, dst)
        linked += 1

total = len(os.listdir(SIM_DIR))
print(f'Symlinked {linked} new files — {total} total in {SIM_DIR}')

In [ ]:
# Verify metadata and normalization files from the repo
import json

with open('dataset/metadata/train.json') as f:
    train_meta = json.load(f)
with open('dataset/metadata/val.json') as f:
    val_meta = json.load(f)

print(f'Train sims: {len(train_meta)}')
print(f'Val   sims: {len(val_meta)}')

# Check how many metadata sims actually have a .pt file in the symlink dir
from pathlib import Path
available = set(os.listdir(SIM_DIR))
train_found = sum(1 for e in train_meta if Path(e['path'].replace('\\','/')).name in available)
val_found   = sum(1 for e in val_meta   if Path(e['path'].replace('\\','/')).name in available)
print(f'Train sims with .pt file: {train_found}/{len(train_meta)}')
print(f'Val   sims with .pt file: {val_found}/{len(val_meta)}')

In [ ]:
# ── FRESH TRAINING ──────────────────────────────────────────────────────────
# Adjust --stride (higher = fewer windows per sim = faster per epoch)
# stride=3 is a good balance: ~3x faster with minimal quality loss

!python src/train.py \
    --dataset_root  dataset \
    --sim_dir       /kaggle/working/simulations \
    --epochs        15 \
    --batch_size    4 \
    --stride        3 \
    --focal \
    --checkpoint    /kaggle/working/models \
    --num_workers   2 \
    --device        auto

In [ ]:
# ── RESUME TRAINING (run this cell instead of the one above after a session ends)
# !python src/train.py \
#     --dataset_root  dataset \
#     --sim_dir       /kaggle/working/simulations \
#     --epochs        15 \
#     --batch_size    4 \
#     --stride        3 \
#     --focal \
#     --checkpoint    /kaggle/working/models \
#     --num_workers   2 \
#     --device        auto \
#     --resume        /kaggle/working/models/latest_model.pt

In [ ]:
# Plot training history
import json, matplotlib.pyplot as plt

with open('/kaggle/working/models/training_history.json') as f:
    history = json.load(f)

epochs     = [r['epoch']    for r in history]
train_loss = [r['train_loss'] for r in history]
val_loss   = [r['val_loss']   for r in history]
val_iou    = [r['val_iou']    for r in history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(epochs, train_loss, label='train'); ax1.plot(epochs, val_loss, label='val')
ax1.set_title('Loss'); ax1.legend(); ax1.set_xlabel('Epoch')
ax2.plot(epochs, val_iou, color='green')
ax2.set_title('Val IoU'); ax2.set_xlabel('Epoch')
plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=150)
plt.show()
print(f'Best val IoU: {max(val_iou):.4f} at epoch {epochs[val_iou.index(max(val_iou))]}')